# Sample 08: 実践業界別レシピ集 (Real-World Industry Recipes)

現場で頻出する4大ドメインの実践的バイナリ設計パターンを体験します。
これまで学んだビットフィールド、オフセット、チェックサム、可変長整数、ストリーミング、多態バリアントを複合的に活用する実用コード集です。

### 収録レシピ
1. **ゲームセーブデータ・アーカイブ**: ヘッダー検証、ビットフラグ、可変オフセットテーブル、データ整合性検証
2. **IoT / 車載テレメトリストリーム**: フレーム同期シグネチャ、Enum、LEB128可変長整数（VarUInt/VarInt）、CRC32
3. **金融ティックロガー (HFT)**: メモリマップファイル（mmap）によるゼロコピー走査、イテレータ処理
4. **多態RPCメッセージキュー**: タグ値に応じた多態ペイロードの自動ディスパッチ（Variant）

## 1. ゲームセーブデータ・アーカイブ形式

ヘッダー検証（`Magic`）、暗号化/圧縮フラグ（`Bits`）、可変個ファイルテーブル（`OffsetTable`）、そしてデータ整合性検証（`CRC32`）を組み合わせた完全なセーブファイル実装です。

In [1]:
from binary_master import (
    BinaryReader,
    BinaryWriter,
    Bits,
    Constant,
    FixedString,
    Magic,
    UInt16,
    UInt32,
    binary_struct,
    compute_checksum,
    hexdump,
    read_struct,
)


# 1. ヘッダービットフラグ (16bit)
@binary_struct(bits=16)
class SaveFlags:
    compressed: Bits[1]
    encrypted: Bits[1]
    hardcore_mode: Bits[1]
    reserved: Bits[13]

# 2. チャンクエントリ
@binary_struct
class PlayerStateChunk:
    player_name: FixedString[16]
    level: UInt16
    hp: UInt32
    gold: UInt32

# 3. メインセーブヘッダー
@binary_struct
class SaveHeader:
    magic: Magic[b"SAVE"]
    version: Constant[UInt16, 1]
    flags: SaveFlags
    chunk_count: UInt16
    chunks_offset: UInt32
    crc32: UInt32

# 書き込み
writer = BinaryWriter()
header = SaveHeader(
    flags=SaveFlags(compressed=0, encrypted=0, hardcore_mode=1),
    chunk_count=2,
    chunks_offset=SaveHeader.binary_size,
    crc32=0,
)
writer.write_struct(header)

# オフセットテーブルの書き込み (2エントリ)
table_handle = writer.write_offset_table(2, offset_size=4)

player1 = PlayerStateChunk(player_name="Hero", level=50, hp=1200, gold=9999)
player2 = PlayerStateChunk(player_name="Mage", level=45, hp=800, gold=5000)

table_handle.set_offset(0, writer.tell())
writer.write_struct(player1)

table_handle.set_offset(1, writer.tell())
writer.write_struct(player2)

# ペイロード全域の CRC32 を計算してヘッダーへバックパッチ
payload = writer.to_bytes()[SaveHeader.binary_size:]
calc_crc = compute_checksum("crc32", payload)
with writer.at_offset(header.offsetof("crc32")):
    writer.write_uint32(calc_crc)

save_data = writer.to_bytes()
print(f"セーブデータ生成完了 ({len(save_data)} バイト):")
print(hexdump(save_data, annotate=True))

# 読み込み & CRC32自動検証
reader = BinaryReader(save_data)
restored_hdr = reader.read_struct(SaveHeader)
assert restored_hdr.magic == b"SAVE"
assert restored_hdr.flags.hardcore_mode == 1

# チェックサムの検証
expected_crc = compute_checksum("crc32", save_data[SaveHeader.binary_size:])
assert restored_hdr.crc32 == expected_crc
print(f"\nCRC32 検証成功: 0x{restored_hdr.crc32:08X}")

reader.seek(restored_hdr.chunks_offset)
offsets = [reader.read_uint32() for _ in range(restored_hdr.chunk_count)]
p1 = read_struct(PlayerStateChunk, save_data[offsets[0]:])
p2 = read_struct(PlayerStateChunk, save_data[offsets[1]:])
print(f"Chunk 0: {p1.player_name}, Lv.{p1.level}, Gold={p1.gold}")
print(f"Chunk 1: {p2.player_name}, Lv.{p2.level}, Gold={p2.gold}")
assert p1.player_name == "Hero"
assert p2.player_name == "Mage"


セーブデータ生成完了 (78 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  53 41 56 45 01 00 04 00  02 00 12 00 00 00 73 e1  |SAVE..........s.|
00000010  c1 df 1a 00 00 00 34 00  00 00 48 65 72 6f 00 00  |......4...Hero..|
00000020  00 00 00 00 00 00 00 00  00 00 32 00 b0 04 00 00  |..........2.....|
00000030  0f 27 00 00 4d 61 67 65  00 00 00 00 00 00 00 00  |.'..Mage........|
00000040  00 00 00 00 2d 00 20 03  00 00 88 13 00 00        |....-. .......  |
  [Total: 78 bytes (`0x004E`)]

CRC32 検証成功: 0xDFC1E173
Chunk 0: Hero, Lv.50, Gold=9999
Chunk 1: Mage, Lv.45, Gold=5000


## 2. IoT / 車載センサーテレメトリストリーム

エッジデバイスからの省帯域・高信頼テレメトリプロトコル。
`Magic`、`BinaryEnum`、LEB128（`VarUInt`, `VarInt`）、および `CRC32` 自動チェックサムを融合します。

In [2]:
from binary_master import CRC32, BinaryEnum, UInt8, VarInt, VarUInt, binary_struct


class SensorType(BinaryEnum):
    TEMPERATURE_HUMIDITY = 1
    ACCELEROMETER = 2
    GPS_LOCATION = 3

@binary_struct
class TelemetryFrame:
    magic: Magic[b"\xAA\x55"]          # 2バイトフレーム同期シグネチャ
    sensor_type: SensorType[UInt8]     # 1バイト列挙型
    device_id: VarUInt                 # LEB128可変長整数 (小さなIDは1B)
    timestamp_delta_ms: VarUInt        # 前フレームからの差分ミリ秒 (可変長)
    reading_delta: VarInt              # 負数対応差分温度 (可変長)
    checksum: CRC32                    # 自動計算 & 検証 CRC32

frame = TelemetryFrame(
    sensor_type=SensorType.TEMPERATURE_HUMIDITY,
    device_id=98765,
    timestamp_delta_ms=16,
    reading_delta=-3,
)
frame_data = frame.to_bytes()
print(f"テレメトリフレーム ({len(frame_data)} バイト): {frame_data.hex()}")

# デシリアライズ & 検証
decoded = TelemetryFrame.from_bytes(frame_data)
print(f"復元結果: Device ID={decoded.device_id}, Type={decoded.sensor_type.name}, Delta={decoded.reading_delta}")
assert decoded.sensor_type == SensorType.TEMPERATURE_HUMIDITY
assert decoded.device_id == 98765
assert decoded.reading_delta == -3
print("IoT テレメトリフレーム検証成功！")


テレメトリフレーム (12 バイト): aa5501cd8306107d45678c6a
復元結果: Device ID=98765, Type=TEMPERATURE_HUMIDITY, Delta=-3
IoT テレメトリフレーム検証成功！


## 3. 高頻度取引 (HFT) / 金融ティックロガー

マイクロ秒精度の市場約定データログ。
GBクラスのファイルでもメモリ使用量を最小に抑えて超高速に走査する **ゼロコピー `from_mmap`** と **`iter_struct`** パターンです。

In [3]:
import tempfile
from pathlib import Path

from binary_master import BinaryReader, Float64, UInt64, binary_struct


@binary_struct
class MarketTick:
    magic: Magic[b"TICK"]
    timestamp_ns: UInt64   # エポックナノ秒
    symbol_id: UInt32      # 銘柄ID
    bid_price: Float64     # 最良買気配
    ask_price: Float64     # 最良売気配
    volume: UInt32         # 約定数量

# サンプル約定ログの作成
ticks = [
    MarketTick(
        timestamp_ns=1700000000000000000 + i * 1_000_000,
        symbol_id=101,
        bid_price=150.0 + i * 0.1,
        ask_price=150.05 + i * 0.1,
        volume=1000 * (i + 1),
    )
    for i in range(5)
]

with tempfile.NamedTemporaryFile(delete=False) as tmp:
    for t in ticks:
        tmp.write(t.to_bytes())
    tmp_path = tmp.name

# 巨大なマーケットログファイルをメモリマップでゼロコピー走査
read_ticks = []
with BinaryReader.from_mmap(tmp_path) as reader:
    # iter_struct はジェネレータで1件ずつゼロコピー復元
    for tick in reader.iter_struct(MarketTick):
        read_ticks.append(tick)
        spread = tick.ask_price - tick.bid_price
        print(f"Tick {tick.symbol_id} at {tick.timestamp_ns}: Bid={tick.bid_price:.2f}, Ask={tick.ask_price:.2f}, Spread={spread:.4f}")

Path(tmp_path).unlink()
assert len(read_ticks) == 5
assert read_ticks[0].bid_price == 150.0
print("金融ティックログ走査完了！")


Tick 101 at 1700000000000000000: Bid=150.00, Ask=150.05, Spread=0.0500
Tick 101 at 1700000000001000000: Bid=150.10, Ask=150.15, Spread=0.0500
Tick 101 at 1700000000002000000: Bid=150.20, Ask=150.25, Spread=0.0500
Tick 101 at 1700000000003000000: Bid=150.30, Ask=150.35, Spread=0.0500
Tick 101 at 1700000000004000000: Bid=150.40, Ask=150.45, Spread=0.0500
金融ティックログ走査完了！


## 4. 多態RPCメッセージキュー

メッセージ種別（`msg_type`）によってペイロード構造が動的に変化するネットワークRPCプロトコル。
`Variant` によるタグベースの自動型ディスパッチです。

In [4]:
from binary_master import BinaryEnum, CString, Variant, binary_struct


class MsgType(BinaryEnum):
    LOGIN_REQ = 1
    CHAT_MSG = 2
    PING = 3

@binary_struct
class LoginPayload:
    user_id: UInt32
    auth_token: CString

@binary_struct
class ChatPayload:
    channel_id: UInt32
    message: CString

@binary_struct
class PingPayload:
    sequence: UInt32

@binary_struct
class RpcMessage:
    magic: Magic[b"RPC\x01"]
    msg_type: MsgType[UInt16]
    body: Variant["msg_type", {
        MsgType.LOGIN_REQ: LoginPayload,
        MsgType.CHAT_MSG: ChatPayload,
        MsgType.PING: PingPayload,
    }]

# 送信パケットの作成 (チャットメッセージ)
msg = RpcMessage(
    msg_type=MsgType.CHAT_MSG,
    body=ChatPayload(channel_id=101, message="Hello Binary Master!"),
)
wire_bytes = msg.to_bytes()
print(f"シリアライズ結果 ({len(wire_bytes)} バイト): {wire_bytes.hex()}")

# 受信側での自動判別・復元
received = RpcMessage.from_bytes(wire_bytes)
print(f"受信メッセージ種別: {received.msg_type.name}")
print(f"ペイロード型: {type(received.body).__name__}")
print(f"メッセージ内容: {received.body.message}")

assert received.msg_type == MsgType.CHAT_MSG
assert isinstance(received.body, ChatPayload)
assert received.body.message == "Hello Binary Master!"
print("多態RPCメッセージの全検証成功！")


シリアライズ結果 (31 バイト): 5250430102006500000048656c6c6f2042696e617279204d61737465722100
受信メッセージ種別: CHAT_MSG
ペイロード型: ChatPayload
メッセージ内容: Hello Binary Master!
多態RPCメッセージの全検証成功！
